In [2]:
# Installing lib 
!pip install plotly openpyxl gradio --quiet
print("Libraries installed!")

Libraries installed!


In [1]:
# Load Text2SQL model
import torch
from unsloth import FastLanguageModel

MODEL_PATH = "/home/jovyan/AnalystAI/models/text2sql-run-1/checkpoint-1100"

print("Loading model...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_PATH,
    max_seq_length = 2048,
    dtype          = torch.float16,
    load_in_4bit   = True,
)
FastLanguageModel.for_inference(model)
print("Model loaded!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/opt/conda/lib/python3.11/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.0' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/conda/lib/python3.11/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.4.0' currently installed).
  from pandas.core import (


Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading model...
==((====))==  Unsloth 2026.3.8: Fast Llama patching. Transformers: 5.3.0.
   \\   /|    NVIDIA L40. Num GPUs = 2. Max memory: 44.394 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.
Unsloth 2026.3.8 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Model loaded!


In [2]:
# Load database and define schema
import sqlite3
import pandas as pd
from pathlib import Path

DB_PATH = Path("/home/jovyan/AnalystAI/database/bikeshare_2024.sqlite")

# Database schema for the model
BIKESHARE_SCHEMA = """CREATE TABLE trips (
    trip_id INTEGER,
    start_time TEXT,
    end_time TEXT,
    duration_minutes REAL,
    start_station INTEGER,
    start_lat REAL,
    start_lon REAL,
    end_station INTEGER,
    end_lat REAL,
    end_lon REAL,
    bike_id INTEGER,
    bike_type TEXT,
    passholder_type TEXT,
    trip_route_category TEXT
);

CREATE TABLE stations (
    station_id TEXT,
    station_name TEXT,
    region TEXT,
    lat REAL,
    lon REAL
);

CREATE TABLE weather (
    timestamp TEXT,
    temp_c REAL,
    rel_humidity REAL,
    precip_mm REAL,
    wind_speed_mps REAL,
    pressure_hpa REAL,
    weather_code REAL
);"""

# Test database connection
conn = sqlite3.connect(DB_PATH)
for table in ['trips', 'stations', 'weather']:
    count = conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"  {table:10} : {count:,} rows")
conn.close()

print("\nDatabase connected!")
print("Schema ready!")

  trips      : 519,235 rows
  stations   : 226 rows
  weather    : 8,784 rows

Database connected!
Schema ready!


In [7]:
# Core functions

import sqlite3
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import torch
import re
from pathlib import Path

DB_PATH = Path("/home/jovyan/AnalystAI/database/bikeshare_2024.sqlite")

# SQL Generation 
def generate_sql(question):
    prompt = f"""### Instruction:
You are an expert SQL assistant. 
Given a database schema and a natural language question, write a correct SQL query. 
Only output the SQL query, no explanation.

### Schema:
{BIKESHARE_SCHEMA}

### Question:
{question}

### SQL:
"""
    inputs = tokenizer(
        [prompt],
        return_tensors     = "pt",
        truncation         = True,
        max_length         = 512,
        add_special_tokens = True,
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            input_ids      = inputs['input_ids'],
            attention_mask = inputs['attention_mask'],
            max_new_tokens = 128,
            do_sample      = False,
            use_cache      = False,
            pad_token_id   = tokenizer.eos_token_id,
        )

    prompt_len = inputs['input_ids'].shape[1]
    generated  = outputs[0][prompt_len:]
    sql = tokenizer.decode(generated, skip_special_tokens=True)
    sql = sql.split("###")[0].strip()
    sql = sql.split("--")[0].strip()
    sql = sql.split("\n\n")[0].strip()
    return sql

# SQL Execution 
def run_sql(sql):
    try:
        conn = sqlite3.connect(DB_PATH)

        # Add MySQL functions to SQLite
        conn.create_function("MONTH", 1,
            lambda x: int(x[5:7]) if x else None)
        conn.create_function("YEAR", 1,
            lambda x: int(x[:4]) if x else None)
        conn.create_function("HOUR", 1,
            lambda x: int(x[11:13]) if x else None)
        conn.create_function("DAY", 1,
            lambda x: int(x[8:10]) if x else None)
        conn.create_function("DATE", 1,
            lambda x: x[:10] if x else None)
        conn.create_function("DAYOFWEEK", 1,
            lambda x: (int(__import__('datetime')
                .datetime.strptime(x[:10], '%Y-%m-%d')
                .strftime('%w')) + 1) if x else None)

        df = pd.read_sql_query(sql, conn)
        conn.close()
        return df, None
    except Exception as e:
        return None, str(e)

# Chart Generation 
def generate_chart(df, question):
    if df is None or len(df) < 2:
        return None

    cols          = df.columns.tolist()
    num_cols      = df.select_dtypes(include='number').columns.tolist()
    cat_cols      = df.select_dtypes(exclude='number').columns.tolist()
    time_keywords = ['month', 'year', 'hour', 'day', 'date', 'week', 'time']

    # Check if ANY column name contains time keyword
    is_time = any(
        k in col.lower()
        for col in cols
        for k in time_keywords
    )

    COLORS = ['#00d2ff', '#7b2ff7', '#ff6b6b', '#ffd93d', '#6bcb77', '#4d96ff']

    layout = dict(
        plot_bgcolor  = '#0d1117',
        paper_bgcolor = '#0d1117',
        font          = dict(color='#e2e8f0', size=12),
        title         = dict(font=dict(size=14, color='#e2e8f0')),
        xaxis         = dict(gridcolor='#1e2530', color='#8892a4'),
        yaxis         = dict(gridcolor='#1e2530', color='#8892a4'),
        margin        = dict(l=50, r=20, t=50, b=50),
    )

    try:
        # Case 1 - single column
        if len(cols) == 1:
            fig = px.histogram(
                df, x=cols[0],
                title                   = question,
                color_discrete_sequence = COLORS
            )

        # Case 2 - time series (line chart)
        elif is_time and len(num_cols) >= 1:
            # Find x col (time) and y col (number)
            x_col = next((c for c in cols if any(
                k in c.lower() for k in time_keywords)), cols[0])
            y_col = num_cols[0]

            # Sort by x for proper line connection
            df_sorted = df.sort_values(x_col).reset_index(drop=True)

            fig = go.Figure()
            fig.add_trace(go.Scatter(
                x         = df_sorted[x_col].tolist(),
                y         = df_sorted[y_col].tolist(),
                mode      = 'lines+markers',
                line      = dict(
                    color = '#00d2ff',
                    width = 3,
                    shape = 'linear',
                ),
                marker    = dict(
                    color = '#7b2ff7',
                    size  = 9,
                    line  = dict(color='#00d2ff', width=2)
                ),
                name      = y_col.replace('_', ' ').title(),
                fill      = 'tozeroy',
                fillcolor = 'rgba(0, 210, 255, 0.08)',
            ))
            fig.update_layout(
                title       = question,
                xaxis_title = x_col.replace('_', ' ').title(),
                yaxis_title = y_col.replace('_', ' ').title(),
            )

        # Case 3 - categorical x, numeric y
        elif len(num_cols) >= 1 and len(cat_cols) >= 1:
            x_col = cat_cols[0]
            y_col = num_cols[0]

            if len(df) <= 6:
                # Pie chart
                fig = px.pie(
                    df,
                    names                   = x_col,
                    values                  = y_col,
                    title                   = question,
                    color_discrete_sequence = COLORS,
                    hole                    = 0.4,
                )
                fig.update_traces(
                    textposition   = 'inside',
                    textinfo       = 'percent+label',
                    textfont_color = 'white',
                )

            elif len(df) <= 15:
                # Vertical bar
                fig = px.bar(
                    df.sort_values(y_col, ascending=False),
                    x                      = x_col,
                    y                      = y_col,
                    title                  = question,
                    color                  = y_col,
                    color_continuous_scale = 'Viridis',
                    labels                 = {
                        x_col: x_col.replace('_', ' ').title(),
                        y_col: y_col.replace('_', ' ').title(),
                    }
                )
            else:
                # Horizontal bar
                fig = px.bar(
                    df.sort_values(y_col, ascending=False).head(15),
                    x                      = y_col,
                    y                      = x_col,
                    orientation            = 'h',
                    title                  = f"{question} (Top 15)",
                    color                  = y_col,
                    color_continuous_scale = 'Plasma',
                    labels                 = {
                        x_col: x_col.replace('_', ' ').title(),
                        y_col: y_col.replace('_', ' ').title(),
                    }
                )

        # Case 4 - two numeric columns
        elif len(num_cols) >= 2:
            # Check if first numeric col is time
            if is_time:
                x_col = num_cols[0]
                y_col = num_cols[1]
                df_sorted = df.sort_values(x_col).reset_index(drop=True)
                fig = go.Figure()
                fig.add_trace(go.Scatter(
                    x         = df_sorted[x_col].tolist(),
                    y         = df_sorted[y_col].tolist(),
                    mode      = 'lines+markers',
                    line      = dict(color='#00d2ff', width=3, shape='linear'),
                    marker    = dict(color='#7b2ff7', size=9,
                                     line=dict(color='#00d2ff', width=2)),
                    name      = y_col.replace('_', ' ').title(),
                    fill      = 'tozeroy',
                    fillcolor = 'rgba(0, 210, 255, 0.08)',
                ))
                fig.update_layout(
                    title       = question,
                    xaxis_title = x_col.replace('_', ' ').title(),
                    yaxis_title = y_col.replace('_', ' ').title(),
                )
            else:
                fig = px.scatter(
                    df,
                    x                       = num_cols[0],
                    y                       = num_cols[1],
                    title                   = question,
                    color_discrete_sequence = COLORS,
                )
        else:
            return None

        fig.update_layout(**layout)
        return fig

    except Exception as e:
        print(f"Chart error: {e}")
        return None

# Insight Generation
def generate_insight(question, df):
    if df is None or len(df) == 0:
        return "No results found for this question."

    num_cols = df.select_dtypes(include='number').columns.tolist()
    cat_cols = df.select_dtypes(exclude='number').columns.tolist()

    # Rule based for single number result
    if len(df) == 1 and len(df.columns) == 1:
        col = df.columns[0]
        val = df.iloc[0, 0]
        if isinstance(val, float):
            return f"The result is {val:,.2f}."
        return f"The result is {int(val):,}."

    # Rule based for single row
    if len(df) == 1:
        parts = []
        for col in df.columns:
            parts.append(f"{col.replace('_',' ')}: {df.iloc[0][col]}")
        return "Result: " + " | ".join(parts)

    # Use Llama for complex results
    summary = df.head(5).to_string(index=False)
    rows    = len(df)

    prompt = f"""### Instruction:
You are a data analyst. 
Given a question and query results, write a 2 sentence insight.
Be specific with numbers. Do not use SQL terms.

### Question:
{question}

### Results ({rows} rows, showing top 5):
{summary}

### Insight:
"""
    inputs = tokenizer(
        [prompt],
        return_tensors     = "pt",
        truncation         = True,
        max_length         = 512,
        add_special_tokens = True,
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            input_ids      = inputs['input_ids'],
            attention_mask = inputs['attention_mask'],
            max_new_tokens = 80,
            do_sample      = False,
            use_cache      = False,
            pad_token_id   = tokenizer.eos_token_id,
        )

    prompt_len = inputs['input_ids'].shape[1]
    generated  = outputs[0][prompt_len:]
    insight = tokenizer.decode(generated, skip_special_tokens=True)
    insight = insight.split("###")[0].strip()
    insight = insight.split("\n\n")[0].strip()
    return insight

print("All functions ready!")

All functions ready!


In [6]:
# Test all functions

test_questions = [
    "How many trips were taken in total?",
    "Which station had the most trips?",
    "How many trips were taken each month?",
]

for q in test_questions:
    print("=" * 50)
    print(f"Question : {q}")

    # Generate SQL
    sql = generate_sql(q)
    print(f"SQL      : {sql}")

    # Run SQL
    df, error = run_sql(sql)
    if error:
        print(f"Error    : {error}")
        continue

    print(f"Rows     : {len(df)}")
    print(f"Data     :\n{df.head(3)}")

    # Generate insight
    insight = generate_insight(q, df)
    print(f"Insight  : {insight}")

    # Generate chart
    fig = generate_chart(df, q)
    if fig:
        fig.show()
        print("Chart    : generated")
    else:
        print("Chart    : not applicable")

print("\nAll tests done!")

Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question : How many trips were taken in total?


/opt/conda/lib/python3.11/site-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/opt/conda/lib/python3.11/site-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/opt/conda/lib/python3.11/site-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API i

SQL      : SELECT COUNT(*) FROM trips;
Rows     : 1
Data     :
   COUNT(*)
0    519235
Insight  : The result is 519,235.
Chart    : not applicable
Question : Which station had the most trips?


/opt/conda/lib/python3.11/site-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/opt/conda/lib/python3.11/site-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


...

Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Rows     : 12
Data     :
   month  trips
0      1  40520
1      2  36065
2      3  43264


/opt/conda/lib/python3.11/site-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/opt/conda/lib/python3.11/site-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


Insight  : The most trips were taken in May with 53108 trips. The least trips were taken in January with 40520 trips.


Chart    : generated

All tests done!


In [10]:
# Gradio App
import gradio as gr
import plotly.express as px
import plotly.graph_objects as go

# ── Analyze function ───────────────────────────────────────────
def analyze(question):
    if not question.strip():
        return "Please enter a question!", None, None, "No insight yet."
    sql       = generate_sql(question)
    df, error = run_sql(sql)
    if error:
        return sql, None, None, f"Could not execute: {error}"
    fig     = generate_chart(df, question)
    insight = generate_insight(question, df)
    return sql, df, fig, insight

def download_csv(question):
    if not question.strip():
        return None
    sql = generate_sql(question)
    df, error = run_sql(sql)
    if error or df is None:
        return None
    path = "/tmp/results.csv"
    df.to_csv(path, index=False)
    return path

def download_excel(question):
    if not question.strip():
        return None
    sql = generate_sql(question)
    df, error = run_sql(sql)
    if error or df is None:
        return None
    path = "/tmp/results.xlsx"
    df.to_excel(path, index=False)
    return path

# ── Chart function ─────────────────────────────────────────────
def generate_chart(df, question):
    if df is None or len(df) < 2:
        return None

    cols          = df.columns.tolist()
    num_cols      = df.select_dtypes(include='number').columns.tolist()
    cat_cols      = df.select_dtypes(exclude='number').columns.tolist()
    time_keywords = ['month', 'year', 'hour', 'day', 'date', 'week', 'time']
    is_time       = any(k in ' '.join(cols).lower() for k in time_keywords)
    COLORS        = ['#00d2ff', '#7b2ff7', '#ff6b6b', '#ffd93d', '#6bcb77', '#4d96ff']

    layout = dict(
        plot_bgcolor  = '#0d1117',
        paper_bgcolor = '#0d1117',
        font          = dict(color='#e2e8f0', size=12),
        title         = dict(font=dict(size=14, color='#e2e8f0')),
        xaxis         = dict(gridcolor='#1e2530', color='#8892a4'),
        yaxis         = dict(gridcolor='#1e2530', color='#8892a4'),
        margin        = dict(l=50, r=20, t=50, b=50),
    )

    try:
        if len(cols) == 1:
            fig = px.histogram(
                df, x=cols[0],
                title                   = question,
                color_discrete_sequence = COLORS
            )

        elif len(num_cols) >= 1 and len(cat_cols) >= 1:
            x_col = cat_cols[0]
            y_col = num_cols[0]

            if is_time or df[x_col].dtype in ['int64', 'float64']:
                # Sort by x for proper line connection
                df_sorted = df.sort_values(x_col).reset_index(drop=True)

                fig = go.Figure()
                fig.add_trace(go.Scatter(
                    x         = df_sorted[x_col].tolist(),  # ← explicit list
                    y         = df_sorted[y_col].tolist(),  # ← explicit list
                    mode      = 'lines+markers',            # ← line + dots
                    line      = dict(
                        color = '#00d2ff',
                        width = 3,
                        shape = 'linear',                   # ← connect dots
                    ),
                    marker    = dict(
                        color = '#7b2ff7',
                        size  = 9,
                        line  = dict(color='#00d2ff', width=2)
                    ),
                    name      = y_col.replace('_', ' ').title(),
                    fill      = 'tozeroy',
                    fillcolor = 'rgba(0, 210, 255, 0.08)',
                ))
                fig.update_layout(
                    title       = question,
                    xaxis_title = x_col.replace('_', ' ').title(),
                    yaxis_title = y_col.replace('_', ' ').title(),
                )

            elif len(df) <= 6:
                # Pie chart for small categories
                fig = px.pie(
                    df,
                    names                   = x_col,
                    values                  = y_col,
                    title                   = question,
                    color_discrete_sequence = COLORS,
                    hole                    = 0.4,
                )
                fig.update_traces(
                    textposition   = 'inside',
                    textinfo       = 'percent+label',
                    textfont_color = 'white',
                )

            elif len(df) <= 15:
                # Vertical bar chart
                fig = px.bar(
                    df.sort_values(y_col, ascending=False),
                    x                      = x_col,
                    y                      = y_col,
                    title                  = question,
                    color                  = y_col,
                    color_continuous_scale = 'Viridis',
                    labels                 = {
                        x_col: x_col.replace('_', ' ').title(),
                        y_col: y_col.replace('_', ' ').title(),
                    }
                )
            else:
                # Horizontal bar for many categories
                fig = px.bar(
                    df.sort_values(y_col, ascending=False).head(15),
                    x                      = y_col,
                    y                      = x_col,
                    orientation            = 'h',
                    title                  = f"{question} (Top 15)",
                    color                  = y_col,
                    color_continuous_scale = 'Plasma',
                    labels                 = {
                        x_col: x_col.replace('_', ' ').title(),
                        y_col: y_col.replace('_', ' ').title(),
                    }
                )

        elif len(num_cols) >= 2:
            fig = px.scatter(
                df,
                x                       = num_cols[0],
                y                       = num_cols[1],
                title                   = question,
                color_discrete_sequence = COLORS,
            )
        else:
            return None

        fig.update_layout(**layout)
        return fig

    except Exception as e:
        print(f"Chart error: {e}")
        return None

# ── CSS ────────────────────────────────────────────────────────
CSS = """
* { box-sizing: border-box; margin: 0; padding: 0; }

body, .gradio-container {
    background: #080c14 !important;
    font-family: 'Inter', 'Segoe UI', sans-serif;
}

.gradio-container::before {
    content: '';
    position: fixed;
    top: 0; left: 0;
    width: 100%; height: 100%;
    background:
        radial-gradient(ellipse at 15% 50%, rgba(123,47,247,0.12) 0%, transparent 50%),
        radial-gradient(ellipse at 85% 20%, rgba(0,210,255,0.08) 0%, transparent 50%),
        radial-gradient(ellipse at 50% 85%, rgba(255,107,107,0.06) 0%, transparent 50%);
    z-index: -1;
}

.app-title {
    font-size: 2.8em;
    font-weight: 800;
    background: linear-gradient(135deg, #00d2ff 0%, #7b2ff7 50%, #ff6b6b 100%);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
    background-clip: text;
    letter-spacing: -1px;
}

.stats-row {
    display: flex;
    gap: 12px;
    padding: 0 20px 15px;
    justify-content: center;
}

.stat-card {
    background: rgba(255,255,255,0.04);
    border: 1px solid rgba(255,255,255,0.08);
    border-radius: 12px;
    padding: 12px 24px;
    text-align: center;
    backdrop-filter: blur(10px);
    transition: transform 0.2s, border-color 0.2s;
}

.stat-card:hover { transform: translateY(-3px); }
.stat-card:nth-child(1) .stat-value { color: #00d2ff; }
.stat-card:nth-child(2) .stat-value { color: #7b2ff7; }
.stat-card:nth-child(3) .stat-value { color: #ffd93d; }
.stat-card:nth-child(4) .stat-value { color: #6bcb77; }

.stat-value {
    font-size: 1.5em;
    font-weight: 700;
}

.stat-label {
    font-size: 0.72em;
    color: #6b7280;
    margin-top: 3px;
    text-transform: uppercase;
    letter-spacing: 0.05em;
}

.ticker-container {
    overflow: hidden;
    background: rgba(255,255,255,0.03);
    border: 1px solid rgba(255,255,255,0.06);
    border-radius: 10px;
    padding: 10px 20px;
    margin: 0 20px 15px;
}

.ticker-label {
    font-size: 0.7em;
    color: #00d2ff;
    text-transform: uppercase;
    letter-spacing: 0.1em;
    font-weight: 600;
    margin-bottom: 4px;
}

.ticker-text {
    color: #e2e8f0;
    font-size: 0.88em;
    animation: tickerScroll 30s linear infinite;
    white-space: nowrap;
    display: inline-block;
}

@keyframes tickerScroll {
    0%   { transform: translateX(0%); }
    100% { transform: translateX(-50%); }
}

.panel-title {
    font-size: 0.72em;
    font-weight: 600;
    text-transform: uppercase;
    letter-spacing: 0.1em;
    margin-bottom: 6px;
}

/* Ask a question - RED color */
.pt-red-ask { color: #ff6b6b; }

.pt-purple { color: #7b2ff7; }
.pt-yellow { color: #ffd93d; }
.pt-green  { color: #6bcb77; }
.pt-red    { color: #ff6b6b; }
.pt-cyan   { color: #00d2ff; }

/* Question input box - BLACK background */
.question-box textarea {
    background: #000000 !important;
    border: 1px solid rgba(255,107,107,0.3) !important;
    border-radius: 10px !important;
    color: #e2e8f0 !important;
    font-size: 0.95em !important;
}

.question-box textarea:focus {
    border-color: #ff6b6b !important;
    box-shadow: 0 0 0 2px rgba(255,107,107,0.15) !important;
}

/* AI Insight box - BLACK background */
.insight-box textarea {
    background: #000000 !important;
    border: 1px solid rgba(255,215,61,0.2) !important;
    border-radius: 10px !important;
    color: #e2e8f0 !important;
    font-size: 0.88em !important;
    line-height: 1.6 !important;
}

.insight-box textarea:focus {
    border-color: #ffd93d !important;
    box-shadow: 0 0 0 2px rgba(255,215,61,0.15) !important;
}

/* All other textareas default */
textarea {
    background: rgba(255,255,255,0.05) !important;
    border: 1px solid rgba(255,255,255,0.1) !important;
    border-radius: 10px !important;
    color: #e2e8f0 !important;
    font-size: 0.95em !important;
}

.analyze-btn button {
    background: linear-gradient(135deg, #7b2ff7 0%, #00d2ff 100%) !important;
    border: none !important;
    border-radius: 10px !important;
    color: white !important;
    font-weight: 700 !important;
    font-size: 1em !important;
    padding: 13px !important;
    width: 100% !important;
    transition: all 0.3s !important;
    letter-spacing: 0.05em !important;
    text-transform: uppercase !important;
}

.analyze-btn button:hover {
    transform: translateY(-2px) !important;
    box-shadow: 0 8px 25px rgba(0,210,255,0.3) !important;
}

.dl-btn-csv button {
    background: rgba(0,210,255,0.1) !important;
    border: 1px solid rgba(0,210,255,0.3) !important;
    color: #00d2ff !important;
    border-radius: 8px !important;
    font-size: 0.82em !important;
    font-weight: 600 !important;
    width: 100% !important;
    transition: all 0.2s !important;
}

.dl-btn-csv button:hover {
    background: rgba(0,210,255,0.2) !important;
}

.dl-btn-excel button {
    background: rgba(107,203,119,0.1) !important;
    border: 1px solid rgba(107,203,119,0.3) !important;
    color: #6bcb77 !important;
    border-radius: 8px !important;
    font-size: 0.82em !important;
    font-weight: 600 !important;
    width: 100% !important;
    transition: all 0.2s !important;
}

.dl-btn-excel button:hover {
    background: rgba(107,203,119,0.2) !important;
}

.gr-code, .gr-code pre, .gr-code code {
    white-space: pre-wrap !important;
    word-break: break-word !important;
    overflow-x: hidden !important;
    background: rgba(255,255,255,0.03) !important;
    border-radius: 8px !important;
    font-size: 0.82em !important;
}

.app-footer {
    text-align: center;
    padding: 15px;
    color: #4a5568;
    font-size: 0.78em;
    border-top: 1px solid rgba(255,255,255,0.05);
    margin-top: 15px;
}

.app-footer a { color: #00d2ff; text-decoration: none; }
.app-footer a:hover { color: #7b2ff7; }
"""

# ── Examples ───────────────────────────────────────────────────
EXAMPLES = [
    "How many trips were taken in total?",
    "How many trips were taken each month?",
    "Which station had the most trips?",
    "What are the top 10 busiest stations?",
    "How many electric vs standard bike trips?",
    "What is the average trip duration by passholder type?",
    "How many trips were taken on rainy days?",
    "What is the busiest hour of the day?",
    "Which region has the most stations?",
    "How many trips were taken on weekends vs weekdays?",
]

ticker_text = "  ✦  ".join(EXAMPLES * 3)

# ── App ────────────────────────────────────────────────────────
with gr.Blocks(title="AnalystAI") as demo:

    gr.HTML(f"""
    <div style="text-align:center; padding:25px 20px 15px;">
        <div class="app-title">AnalystAI</div>
        <div style="color:#8892a4; font-size:0.9em; margin-top:6px;">
            LA Metro Bike Share 2024 &nbsp;·&nbsp;
            AI-Powered Data Analyst &nbsp;·&nbsp;
            Fine-tuned Llama 3.2 3B
        </div>
    </div>
    """)

    gr.HTML("""
    <div class="stats-row">
        <div class="stat-card">
            <div class="stat-value">519K</div>
            <div class="stat-label">Total Trips</div>
        </div>
        <div class="stat-card">
            <div class="stat-value">226</div>
            <div class="stat-label">Stations</div>
        </div>
        <div class="stat-card">
            <div class="stat-value">2024</div>
            <div class="stat-label">Data Year</div>
        </div>
        <div class="stat-card">
            <div class="stat-value">Llama 3.2</div>
            <div class="stat-label">AI Model</div>
        </div>
    </div>
    """)

    gr.HTML(f"""
    <div class="ticker-container">
        <div class="ticker-label">Try Asking</div>
        <div style="overflow:hidden;">
            <div class="ticker-text">{ticker_text}</div>
        </div>
    </div>
    """)

    with gr.Row():

        # Left panel
        with gr.Column(scale=1):

            # Ask a question - RED title
            gr.HTML('<div class="panel-title pt-red-ask">Ask a Question</div>')
            question_input = gr.Textbox(
                placeholder  = "e.g. How many trips were taken each month?",
                lines        = 3,
                show_label   = False,
                elem_classes = ["question-box"],  # ← black background
            )

            analyze_btn = gr.Button(
                "Analyze",
                variant      = "primary",
                elem_classes = ["analyze-btn"],
            )

            with gr.Row():
                csv_btn = gr.DownloadButton(
                    "Download CSV",
                    variant      = "secondary",
                    elem_classes = ["dl-btn-csv"],
                )
                excel_btn = gr.DownloadButton(
                    "Download Excel",
                    variant      = "secondary",
                    elem_classes = ["dl-btn-excel"],
                )

            gr.HTML('<div class="panel-title pt-purple" style="margin-top:12px;">Results Table</div>')
            table_output = gr.Dataframe(
                wrap       = True,
                show_label = False,
            )

        # Right panel
        with gr.Column(scale=1):

            gr.HTML('<div class="panel-title pt-green">Generated SQL Query</div>')
            sql_output = gr.Code(
                language   = "sql",
                lines      = 5,
                show_label = False,
            )

            # AI Insight - BLACK background
            gr.HTML('<div class="panel-title pt-yellow" style="margin-top:12px;">AI Insight</div>')
            insight_output = gr.Textbox(
                lines        = 3,
                interactive  = False,
                show_label   = False,
                elem_classes = ["insight-box"],  # ← black background
            )

            gr.HTML('<div class="panel-title pt-red" style="margin-top:12px;">Chart</div>')
            chart_output = gr.Plot(show_label=False)

    gr.HTML("""
    <div class="app-footer">
        Built by Krupa Patel &nbsp;|&nbsp;
        <a href="https://huggingface.co/Krupa1420/text2sql-llama3-qlora">Model</a>
        &nbsp;|&nbsp;
        <a href="https://huggingface.co/datasets/Krupa1420/la-metro-bikeshare-2024">Dataset</a>
        &nbsp;|&nbsp;
        MS CS — Cal State Northridge
    </div>
    """)

    # Wire up
    analyze_btn.click(
        fn      = analyze,
        inputs  = [question_input],
        outputs = [sql_output, table_output, chart_output, insight_output],
    )
    question_input.submit(
        fn      = analyze,
        inputs  = [question_input],
        outputs = [sql_output, table_output, chart_output, insight_output],
    )
    csv_btn.click(
        fn      = download_csv,
        inputs  = [question_input],
        outputs = [csv_btn],
    )
    excel_btn.click(
        fn      = download_excel,
        inputs  = [question_input],
        outputs = [excel_btn],
    )

print("Launching AnalystAI...")
demo.launch(share=True, show_error=True, css=CSS)

Launching AnalystAI...
* Running on local URL:  http://127.0.0.1:7867
* Running on public URL: https://b35782126c00e5cb93.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/opt/conda/lib/python3.11/site-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/opt/conda/lib/python3.11/site-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=

In [ ]:
import os
import subprocess
from pathlib import Path

BASE = Path("/home/jovyan/AnalystAI")
os.chdir(BASE)

def run(cmd):
    result = subprocess.run(
        cmd, shell=True,
        capture_output=True, text=True
    )
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)

# Create README
readme = """# AnalystAI: AI Powered Data Analyst

> Ask questions in plain English about LA Metro Bike Share 2024 data and get SQL queries, interactive charts, and AI insights instantly.


## What is AnalystAI?

AnalystAI is an end-to-end AI powered data analyst built on top of a fine-tuned Llama 3.2 3B model. It eliminates the need for SQL knowledge — users simply ask questions in plain English and instantly receive:

- Auto-generated SQL queries
- Interactive charts (bar, line, pie)
- AI-generated data insights
- Downloadable results (CSV and Excel)

## Project Structure
```
AnalystAI/
  notebooks/
    t2s_dataset_creation.ipynb   Text2SQL training dataset creation
    t2s_training.ipynb           QLoRA fine-tuning of Llama 3.2 3B
    t2s_evaluation.ipynb         Model evaluation on Spider benchmark
    bs_data_preparation.ipynb    LA Metro Bike Share data preparation
    analyst_app.ipynb            Main Gradio application
  database/
    README.md                    Database schema documentation
```

## Tech Stack

| Component | Technology |
|-----------|------------|
| LLM | Llama 3.2 3B (fine-tuned with QLoRA) |
| Training | Unsloth + TRL + PEFT |
| Database | SQLite |
| Charts | Plotly |
| App | Gradio |
| Data | LA Metro Bike Share 2024 |

## Model

Fine-tuned Llama 3.2 3B using QLoRA on 7,300 Text-to-SQL examples combining Spider benchmark and LA Metro Bike Share domain-specific queries.


**Model on HuggingFace:** [Krupa1420/text2sql-llama3-qlora](https://huggingface.co/Krupa1420/text2sql-llama3-qlora)

## Dataset

Real LA Metro Bike Share 2024 data cleaned and published on HuggingFace.

| Table | Rows | Description |
|-------|------|-------------|
| trips | 519,235 | All bike trips in 2024 |
| stations | 226 | Active station locations |
| weather | 8,784 | Hourly LA weather data |

**Dataset on HuggingFace:** [Krupa1420/la-metro-bikeshare-2024](https://huggingface.co/datasets/Krupa1420/la-metro-bikeshare-2024)

**Text2SQL Training Dataset:** [Krupa1420/text2sqldata](https://huggingface.co/datasets/Krupa1420/text2sqldata)

## Example Queries

| Question | Type |
|----------|------|
| How many trips were taken in total? | Simple |
| How many trips were taken each month? | Line Chart |
| What are the top 10 busiest stations? | Bar Chart |
| How many electric vs standard bike trips? | Pie Chart |
| Which station had the most trips on rainy days? | Triple JOIN |
| What is the average trip duration on rainy vs sunny days? | Weather JOIN |

## How to Run

### Step 1 - Clone repo
```bash
git clone https://github.com/Krupa1420/AnalystAI.git
cd AnalystAI
```

### Step 2 - Install dependencies
```bash
pip install unsloth gradio plotly pandas openpyxl torch transformers
```

### Step 3 - Download database
```python
from huggingface_hub import hf_hub_download

db_path = hf_hub_download(
    repo_id   = "Krupa1420/la-metro-bikeshare-2024",
    filename  = "bikeshare_2024.sqlite",
    repo_type = "dataset"
)
```

### Step 4 - Run the app
Open `notebooks/analyst_app.ipynb` and run all cells.

## Architecture
```
User Question (plain English)
        |
Fine-tuned Llama 3.2 3B (Text-to-SQL)
        |
SQL Query generated
        |
SQLite Database (519K real trips)
        |
Results + Chart + AI Insight
```

## Author

**Krupa Patel**
MS Computer Science — California State University Northridge

[![HuggingFace](https://img.shields.io/badge/HuggingFace-Krupa1420-yellow)](https://huggingface.co/Krupa1420)
[![GitHub](https://img.shields.io/badge/GitHub-Krupa1420-black)](https://github.com/Krupa1420)
"""

with open(BASE / "README.md", "w") as f:
    f.write(readme)
print("README created!")

# Create .gitignore
gitignore = """
# Large data files
data/
models/
cache/
database/bikeshare_2024.sqlite
database/trips.csv
database/weather.csv
database/stations.csv

# Archives
*.zip
*.tar.gz

# Model files
*.pt
*.safetensors
*.bin

# Python
__pycache__/
*.pyc
*.pyo
.ipynb_checkpoints/
unsloth_compiled_cache/

# System
.DS_Store
*.log
"""

with open(BASE / ".gitignore", "w") as f:
    f.write(gitignore)
print(".gitignore created!")

# Remove tokens from notebooks
import re
import json

notebooks = [
    "notebooks/t2s_dataset_creation.ipynb",
    "notebooks/t2s_training.ipynb",
    "notebooks/t2s_evaluation.ipynb",
    "notebooks/bs_data_preparation.ipynb",
    "notebooks/analyst_app.ipynb",
]

print("\nRemoving tokens from notebooks...")
for nb in notebooks:
    path = BASE / nb
    if not path.exists():
        print(f"  Not found: {nb}")
        continue
    with open(path, 'r') as f:
        content = f.read()
    content = re.sub(r'hf_[A-Za-z0-9]+', 'YOUR_HF_TOKEN_HERE', content)
    with open(path, 'w') as f:
        f.write(content)
    print(f"  Token removed: {path.name}")

print("\nSetting up git...")
run('git config --global user.email "krupapatel1420@gmail.com"')
run('git config --global user.name "Krupa1420"')
run("git init")
run("git branch -M main")

# Add files
print("\nAdding files...")
run("git add README.md")
run("git add .gitignore")
run("git add notebooks/t2s_dataset_creation.ipynb")
run("git add notebooks/t2s_training.ipynb")
run("git add notebooks/t2s_evaluation.ipynb")
run("git add notebooks/bs_data_preparation.ipynb")
run("git add notebooks/analyst_app.ipynb")
run("git add database/README.md")

# Commit
run('git commit -m "AnalystAI: AI-Powered Data Analyst with fine-tuned Llama 3.2 3B on LA Metro Bike Share 2024"')

print("\nGit commit done!")
print("Now create repo on GitHub then run the push cell below!")